# AppleAlt: ResNet18 / VGG16 / VGG19 on Colab GPU
**Scope:** train exactly **ResNet18, VGG16 and VGG19** here. **MobileNetV2 was trained on the PC** and is not retrained or touched.

**Protocol (identical to the local MobileNetV2 run):** 5 epochs | batch 32 (reduced only on GPU out-of-memory, recorded in `config.json`) | Adam lr 1e-4 | seed 42 | weighted cross-entropy | ImageNet preprocessing | best checkpoint chosen by **validation accuracy only** (tie-break lowest val loss) | the test manifest is **never opened** here.

**Data:** the finalized, group-aware manifests (`results/retrained/{train,val}_manifest.csv`, SHA-256 verified). Splits, class mapping (`Healthy=0, Alternaria=1`), architectures and the evaluation protocol are **not** changed.

**Before running:** upload to Google Drive `MyDrive/apple_leaf_retrained/upload/`:
1. `apple_leaf_project_files.zip` (code + manifests, ~2 MB)
2. `apple_leaf_dataset.zip` (the two `DATASET/` folders, ~1.0 GB, stored uncompressed)

Runtime: *Runtime > Change runtime type > T4 GPU*. Outputs sync to `MyDrive/apple_leaf_retrained/outputs/` every 2 minutes and after each model; re-running after a disconnect **resumes** from the per-epoch rolling checkpoint or **skips** completed models. Evaluation is **not** run on Colab: all four checkpoints are evaluated once on the PC.

In [ ]:
# 0. GPU check
import torch, subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip() or "nvidia-smi unavailable")
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Select a GPU runtime (Runtime > Change runtime type > T4 GPU) before continuing."


In [ ]:
# 1. Configuration and Drive mount
import os, sys, json, time, shutil, hashlib, threading, zipfile, runpy, glob
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive/apple_leaf_retrained"
UPLOAD_DIR = f"{DRIVE_ROOT}/upload"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs"                 # persistent copies of models/ and results/
PROJECT = "/content/apple_leaf"                       # fast local working copy (same layout as the PC project)
MODELS = ["resnet18", "vgg16", "vgg19"]   # MobileNetV2 was trained on the PC and is NOT retrained here
EPOCHS, BATCH, NUM_WORKERS = 5, 32, 2
BATCH_FALLBACKS = [32, 16, 8]              # tried in order only if the GPU runs out of memory (recorded in config.json)
RUN_EVALUATION_ON_COLAB = False   # project decision: evaluate ONCE, locally on the deployment CPU, after all checkpoints are back

EXPECTED_MANIFEST_SHA256 = {
    "train_manifest.csv": "0822ad5ce148d7ef67b3392a6da5ed7216cea08f9339d07a2839b7929ff2a112",
    "val_manifest.csv": "93eae293c450907c42f8edea11f355dbcebec45cf4a10a6e99d2c454ecb9c9fc",
    "test_manifest.csv": "6e60b832f3a6d07c57fcb0bb4b5ffa12ea632bf71a84ced16557063b7eb35735"
}
EXPECTED_IMAGE_COUNTS = {"Alternaria leaf spot": 10265, "Healthy": 11926}
os.makedirs(OUTPUT_DIR, exist_ok=True)
for z in ("apple_leaf_project_files.zip", "apple_leaf_dataset.zip"):
    assert os.path.exists(f"{UPLOAD_DIR}/{z}"), f"Missing {UPLOAD_DIR}/{z} - upload it first."
print("Drive OK:", os.listdir(UPLOAD_DIR))


In [ ]:
# 2. Unpack project files + dataset to local disk and VERIFY them against the finalized manifests
os.makedirs(PROJECT, exist_ok=True)
with zipfile.ZipFile(f"{UPLOAD_DIR}/apple_leaf_project_files.zip") as z:
    z.extractall(PROJECT)
if not os.path.isdir(f"{PROJECT}/DATASET/Healthy"):
    t = time.time()
    with zipfile.ZipFile(f"{UPLOAD_DIR}/apple_leaf_dataset.zip") as z:
        z.extractall(PROJECT)
    print(f"dataset extracted in {time.time()-t:.0f}s")
os.makedirs(f"{PROJECT}/models/retrained", exist_ok=True)
os.makedirs(f"{PROJECT}/results/retrained_evaluation/curves", exist_ok=True)

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
for folder, n in EXPECTED_IMAGE_COUNTS.items():
    got = sum(1 for f in os.listdir(f"{PROJECT}/DATASET/{folder}") if os.path.splitext(f)[1].lower() in IMG_EXT)
    assert got == n, f"{folder}: {got} images, expected {n}"
    print(f"{folder}: {got} images OK")
for f, h in EXPECTED_MANIFEST_SHA256.items():
    actual = hashlib.sha256(open(f"{PROJECT}/results/retrained/{f}", "rb").read()).hexdigest()
    assert actual == h, f"{f} differs from the finalized manifest (sha256 {actual})"
    print(f"{f}: sha256 OK")

# full byte-level check: every image in the three manifests must match its recorded sha256
import pandas as pd
man = pd.concat([pd.read_csv(f"{PROJECT}/results/retrained/{s}_manifest.csv") for s in ("train", "val", "test")])
t = time.time(); bad = []
for i, (rel, sha) in enumerate(zip(man.relative_path, man.sha256)):
    with open(f"{PROJECT}/{rel}", "rb") as fh:
        if hashlib.sha256(fh.read()).hexdigest() != sha:
            bad.append(rel)
    if (i + 1) % 5000 == 0:
        print(f"  hashed {i+1}/{len(man)} ...")
assert not bad, f"{len(bad)} images differ from the manifest, e.g. {bad[:3]}"
print(f"all {len(man)} split images match their manifest sha256 ({time.time()-t:.0f}s)")


In [ ]:
# 3. Restore previous outputs from Drive (enables resume / skip) and start the background sync to Drive
SYNC_PATHS = ["models/retrained", "results/retrained_evaluation"] + [f"results/retrained/{m}" for m in MODELS]

def copy_tree(src, dst):
    if not os.path.isdir(src):
        return 0
    n = 0
    for root, _, files in os.walk(src):
        for f in files:
            s = os.path.join(root, f); d = os.path.join(dst, os.path.relpath(s, src))
            if not os.path.exists(d) or os.path.getmtime(s) > os.path.getmtime(d) + 1 or os.path.getsize(s) != os.path.getsize(d):
                os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copy2(s, d); n += 1
    return n

restored = sum(copy_tree(f"{OUTPUT_DIR}/{p}", f"{PROJECT}/{p}") for p in SYNC_PATHS)
print("files restored from Drive:", restored, "|", sorted(os.listdir(f"{PROJECT}/models/retrained")))

def sync_to_drive():
    return sum(copy_tree(f"{PROJECT}/{p}", f"{OUTPUT_DIR}/{p}") for p in SYNC_PATHS)

_stop = threading.Event()
def _syncer():
    while not _stop.wait(120):
        try:
            n = sync_to_drive()
            if n: print(f"[sync] {n} file(s) copied to Drive at {time.strftime('%H:%M:%S')}")
        except Exception as e:
            print("[sync] warning:", e)
threading.Thread(target=_syncer, daemon=True).start()
print("background sync to Drive every 120 s started")


In [ ]:
# 4. Train the four models sequentially (same script, same manifests; --resume makes re-runs safe)
os.chdir(PROJECT)
if PROJECT + "/code" not in sys.path:
    sys.path.insert(0, PROJECT + "/code")

test_opens = []
sys.addaudithook(lambda ev, a: test_opens.append(a[0]) if ev == "open" and a and isinstance(a[0], str) and "test_manifest" in a[0].lower() else None)

def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or ("out of memory" in str(exc).lower() and isinstance(exc, RuntimeError))

status = {}
for m in MODELS:
    t0 = time.time()
    print(f"\n{'#'*70}\n# {m.upper()}  ({EPOCHS} epochs)  {time.strftime('%Y-%m-%d %H:%M:%S')}\n{'#'*70}", flush=True)
    for bs in BATCH_FALLBACKS:
        sys.argv = ["train_comparative_models.py", "--model", m, "--epochs", str(EPOCHS), "--batch-size", str(bs),
                    "--num-workers", str(NUM_WORKERS), "--resume"]   # --resume: continues from the rolling checkpoint after any interruption
        try:
            runpy.run_path(f"{PROJECT}/code/train_comparative_models.py", run_name="__main__")
            status[m] = f"completed (batch {bs})" + ("" if bs == BATCH else "  <-- batch size reduced after GPU OOM; noted in config.json")
            break
        except Exception as exc:
            if is_oom(exc) and bs != BATCH_FALLBACKS[-1]:
                print(f"GPU out of memory at batch {bs}; retrying with the next smaller batch size (progress is kept by the rolling checkpoint)", flush=True)
                torch.cuda.empty_cache(); import gc; gc.collect()
                continue
            status[m] = f"FAILED: {type(exc).__name__}: {exc}"
            print(status[m]); break
    n = sync_to_drive(); print(f"[sync] {n} file(s) -> Drive | {m}: {status[m]} | {(time.time()-t0)/60:.1f} min", flush=True)
    torch.cuda.empty_cache()
print("\nstatus:", status)
print("test manifest opened during training:", test_opens)
assert not test_opens, "Test manifest was opened during training - investigate before evaluating."


In [ ]:
# 5. Verify training artefacts (5 epochs, finite metrics, best epoch = max val acc, manifest hashes)
import numpy as np
problems = []
for m in MODELS:
    out = f"{PROJECT}/results/retrained/{m}"; ck = f"{PROJECT}/models/retrained/{m}_best.pth"
    try:
        h = pd.read_csv(f"{out}/history.csv"); cfg = json.load(open(f"{out}/config.json"))
        ok = (len(h) == EPOCHS and np.isfinite(h[["train_loss","val_loss","train_acc","val_acc"]].values).all()
              and abs(cfg["best_val_accuracy"] - h.val_acc.max()) < 1e-6 and os.path.exists(ck)
              and cfg["train_manifest_sha256"] == EXPECTED_MANIFEST_SHA256["train_manifest.csv"]
              and cfg["val_manifest_sha256"] == EXPECTED_MANIFEST_SHA256["val_manifest.csv"]
              and cfg["test_manifest_used_during_training"] is False)
        print(f"{m:12s} epochs={len(h)} best_epoch={cfg['best_epoch']} best_val_acc={cfg['best_val_accuracy']*100:.2f}% "
              f"train_acc@last={h.train_acc.iloc[-1]*100:.2f}% val_loss {h.val_loss.iloc[0]:.4f}->{h.val_loss.iloc[-1]:.4f} | {'OK' if ok else 'PROBLEM'}")
        if not ok: problems.append(m)
    except Exception as e:
        problems.append(m); print(m, "missing/invalid:", e)
assert not problems, f"Fix before evaluating: {problems}"
print("all four models verified")


In [ ]:
# 6. Evaluate ONCE on the held-out test manifest (full split + eval_eligible real-image subset)
EVAL_DIR = f"{PROJECT}/results/retrained_evaluation"
if not RUN_EVALUATION_ON_COLAB:
    print("RUN_EVALUATION_ON_COLAB is False: skipping. Download the checkpoints and run, on the PC:\n"
          "    python code/evaluate_all_models.py\n(one evaluation on the deployment CPU, so latency columns are meaningful).")
elif os.path.exists(f"{OUTPUT_DIR}/results/retrained_evaluation/comparison.csv"):
    print("Evaluation results already exist on Drive - not re-running (the test set is evaluated once). Delete that folder deliberately if a re-run is truly needed.")
    copy_tree(f"{OUTPUT_DIR}/results/retrained_evaluation", EVAL_DIR)
else:
    sys.argv = ["evaluate_all_models.py", "--device", "cuda", "--batch-size", "32"]
    runpy.run_path(f"{PROJECT}/code/evaluate_all_models.py", run_name="__main__")
    print("[sync]", sync_to_drive(), "file(s) -> Drive")


In [ ]:
# 7. Results summary (report all four models; do not pick a model by test accuracy)
pd.set_option("display.width", 220)
if not os.path.exists(f"{EVAL_DIR}/comparison.csv"):
    print("No evaluation results on Colab (evaluation is run on the PC). Training outputs to download:")
    for p in SYNC_PATHS: print("  ", f"{OUTPUT_DIR}/{p}  ->  <project>/{p}")
    raise SystemExit(0)
test = pd.read_csv(f"{PROJECT}/results/retrained/test_manifest.csv"); ee = test[test.eval_eligible]
print(f"full test split: {len(test)} images / {test.group_id.nunique()} groups | eval_eligible: {len(ee)} images / {ee.group_id.nunique()} groups")
cols = ["model_name","test_samples","test_accuracy","precision_healthy","recall_healthy","f1_healthy","precision_alternaria","recall_alternaria","f1_alternaria","f1_macro","confusion_matrix"]
for name in ("comparison.csv", "comparison_eval_eligible.csv"):
    d = pd.read_csv(f"{EVAL_DIR}/{name}")[cols].copy()
    for c in cols[2:-1]: d[c] = (d[c]*100).round(2)
    print(f"\n=== {name} (metrics in %) ===\n" + d.to_string(index=False))
for m in MODELS:
    cfg = json.load(open(f"{PROJECT}/results/retrained/{m}/config.json")); f = pd.read_csv(f"{EVAL_DIR}/comparison.csv").set_index("model_name").loc[m]
    print(f"{m:12s} val(best) {cfg['best_val_accuracy']*100:6.2f}%  test(full) {f.test_accuracy*100:6.2f}%  gap {(cfg['best_val_accuracy']-f.test_accuracy)*100:+.2f} pts")
print("\nNote: latency/throughput columns were measured on the Colab GPU, not the deployment CPU.")
print("\nDownload back to the PC project (same relative paths):")
for p in SYNC_PATHS: print("  ", f"{OUTPUT_DIR}/{p}  ->  <project>/{p}")


## Bringing results back to the PC
Copy from `MyDrive/apple_leaf_retrained/outputs/` into the local project **at the same relative paths**:
- `models/retrained/<model>_best.pth` (loaded by the Streamlit app, the FastAPI backend and `04_quick_inference.py`; they load with `map_location`, so GPU-saved checkpoints work on CPU)
- `results/retrained/<model>/` (history, curves, config, log)
- `results/retrained_evaluation/` (comparison tables, per-class reports, confusion matrices, predictions)

Then run locally `python code/prepare_clean_splits.py --validate-only --no-decode` (manifests unchanged) and verify the checkpoints load. Do **not** update the website's displayed metrics until the evaluation outputs have been reviewed.